# NLI Models Script

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
nli_dir = '/content/drive/MyDrive/nli'
os.makedirs(nli_dir, exist_ok=True)
os.chdir(nli_dir)

print(os.getcwd())

Mounted at /content/drive/
/content/drive/MyDrive/nli


In [2]:
# Create dict of codes

import re
from config import I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1

blocks = {
    "I_1": I_1, "I_2": I_2, "I_3": I_3, "I_4": I_4, "I_5": I_5, "I_NO_1": I_NO_1,
    "N_1": N_1, "N_2": N_2, "N_NO_1": N_NO_1,
    "J_1": J_1, "J_2": J_2, "J_3": J_3, "J_NO_1": J_NO_1,
}

CODE_PATTERN = re.compile(
    r'-\s*([A-Z0-9]+)\s*\n?\s*"(.*?)"\s*(?=\s*(?:\d+\.\d+\s+[A-Z0-9]+\s*)?-\s*[A-Z0-9]+|\s*\Z)',
    re.MULTILINE | re.DOTALL
)

codebook = {}
for name, block_text in blocks.items():
    for match in CODE_PATTERN.finditer(block_text):
        code_id, definition = match.groups()
        codebook[code_id] = definition.replace('\\"', '"').strip()

print(f"Parsed {len(codebook)} codes")

Parsed 127 codes


In [3]:
import pandas as pd
judaism = pd.read_csv("ucberkeley-dlab_target_jewish.csv")
code_features = pd.read_csv("code_features.csv").set_index("comment_id")
print(judaism.shape)
print(code_features.shape)

(1874, 3)
(1874, 381)


In [9]:
from transformers import pipeline
import torch

nli_pipe = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v1",
    device=0
)

print(nli_pipe.device)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

cuda:0


In [5]:
# small test
test_code = "N1BELIEFS"
test_definition = codebook[test_code]

sample_texts = judaism["text"].head(5).tolist()

for text in sample_texts:
    result = nli_pipe(text, candidate_labels=[test_definition], hypothesis_template="This text expresses: {}.")
    print(f"\nText: {text[:100]}")
    print(f"Entailment score for '{test_code}': {result['scores'][0]:.3f}")


Text: Because it's the Word of God maybe?  Just spitballin
Entailment score for 'N1BELIEFS': 0.000

Text: @Cochis3 Not all the public, including Jews, are stupid enough to believe the absolute effluent spew
Entailment score for 'N1BELIEFS': 1.000

Text: I can't believe that this would be problematic for anyone. Of all the things there is to worry about
Entailment score for 'N1BELIEFS': 0.870

Text: I have a solution. Send them back to where they came from. THEY CAUSED THEIR OWN PROBLEMS by choosin
Entailment score for 'N1BELIEFS': 0.963

Text: OY VEY! The "Jewish Race" is so smart, they worship Lucifer & Saturn, and get their rocks off on Ins
Entailment score for 'N1BELIEFS': 1.000


In [11]:
# Score full pilot corpus against all codes
import csv
import os
import torch
from tqdm.auto import tqdm

output_path = "nli_zeroshot_scores.csv"
BATCH_SIZE = 96  # tune if OOM on L4

texts = judaism["text"].tolist()
comment_ids = judaism["comment_id"].tolist()
code_ids = list(codebook.keys())
definitions = [codebook[c] for c in code_ids]

already_done = set()
if os.path.exists(output_path):
    existing = pd.read_csv(output_path)
    already_done = set(existing["code_id"].unique())
    print(f"Resuming: {len(already_done)} codes already scored, skipping those")

remaining = [(c, d) for c, d in zip(code_ids, definitions) if c not in already_done]

file_exists = os.path.exists(output_path)
with open(output_path, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["comment_id", "code_id", "entailment_score"])

    pbar = tqdm(remaining, desc="Scoring codes", unit="code")
    for code_id, definition in pbar:
        pbar.set_postfix(code=code_id)

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = nli_pipe(
                texts,
                candidate_labels=[definition],
                hypothesis_template="This text expresses: {}.",
                multi_label=True,
                batch_size=BATCH_SIZE,
                truncation=True,
            )

        if isinstance(outputs, dict):
            outputs = [outputs]
        all_scores = [o["scores"][0] for o in outputs]

        for cid, score in zip(comment_ids, all_scores):
            writer.writerow([cid, code_id, score])
        f.flush()

print("All codes scored.")

Resuming: 0 codes already scored, skipping those


Scoring codes:   0%|          | 0/127 [00:00<?, ?code/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


All codes scored.
